In [1]:
import datetime as dt

from sqlalchemy import create_engine
from sqlalchemy import func
from sqlalchemy import update
from sqlalchemy.orm import Session

import src
from src.data.models import Comment
from src.data.models import Sentence
from src.data.models import Video

In [2]:
engine = create_engine(src.PS_ENGINE)

# Reset all exclusions

In [6]:
with Session(engine) as s:
    s.execute(update(Video).values(is_valid=True))
    s.execute(update(Sentence).values(is_valid=True))
    s.execute(update(Comment).values(is_valid=True))
    s.commit()
    print(s.query(Video).filter(Video.is_valid == False).count())
    print(s.query(Sentence).filter(Sentence.is_valid == False).count())

0
0


# Exclusions

## Sentence is too short

In [8]:
with Session(engine) as s:
    s.execute(
        update(Sentence).where(func.array_length(Sentence.tokens, 1) < 5).values(is_valid=False),
    )
    s.commit()

In [9]:
with Session(engine) as s:
    count = s.query(Sentence).filter(Sentence.is_valid == False).count()
    print(count)

62815


## Video has less than 5 valid sentences

In [10]:
with Session(engine) as s:
    sent_lengths = (
        s.query(Video)
        .join(Sentence)
        .group_by(Video.id)
        .filter(Sentence.is_valid == True)
        .with_entities(Video.id, func.count(Video.id).label("sent_count"))
        .subquery()
    )
    too_short = (
        s.query(sent_lengths).filter(sent_lengths.c.sent_count < 5).with_entities(sent_lengths.c.id)
    )
    long_enough = s.query(sent_lengths).filter(sent_lengths.c.sent_count >= 5)

    print(too_short.count(), long_enough.count(), too_short.count() + long_enough.count())

    s.execute(update(Video).where(Video.id.in_(too_short)).values(is_valid=False))
    s.commit()

814 16143 16957


In [11]:
with Session(engine) as s:
    count = s.query(Video).filter(Video.is_valid == False).count()
    print(count)

814


## Video is from before 2017-12-06

In [12]:
with Session(engine) as s:
    s.execute(
        update(Video)
        .where(Video.datetime_upload <= dt.datetime(2017, 12, 6))
        .values(is_valid=False),
    )
    s.commit()

In [13]:
with Session(engine) as s:
    count = s.query(Video).filter(Video.is_valid == False).count()
    print(count)

7323


## Video is a Short

In [14]:
with Session(engine) as s:
    s.execute(
        update(Video)
        .where(Video.format != "videos")
        .values(is_valid=False),
    )
    s.commit()

In [15]:
with Session(engine) as s:
    count = s.query(Video).filter(Video.is_valid == False).count()
    print(count)

7860


## Sentence occurs in invalid video

In [16]:
with Session(engine) as s:
    count = s.query(Sentence).filter(Sentence.is_valid == False).count()
    print(count)

62815


In [17]:
with Session(engine) as s:
    invalid_videos = (
        s.query(Video).filter(Video.is_valid == False).with_entities(Video.id)
    )
    s.execute(update(Sentence).where(Sentence.video_id.in_(invalid_videos)).values(is_valid=False))
    s.commit()

In [18]:
with Session(engine) as s:
    count = s.query(Sentence).filter(Sentence.is_valid == False).count()
    print(count)

587857


## Comment is from an invalid video

In [3]:
with Session(engine) as s:
    count = s.query(Comment).filter(Comment.is_valid == False).count()
    print(count)

0


In [4]:
with Session(engine) as s:
    invalid_videos = (
        s.query(Video).filter(Video.is_valid == False).with_entities(Video.id)
    )
    s.execute(update(Comment).where(Comment.video_id.in_(invalid_videos)).values(is_valid=False))
    s.commit()

In [5]:
with Session(engine) as s:
    count = s.query(Comment).filter(Comment.is_valid == False).count()
    print(count)

230555
